In [79]:
%pip install langchain langchain-openai langsmith --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [86]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.environ["LANGSMITH_TRACING"]
os.environ["LANGCHAIN_PROJECT"] = os.environ["LANGSMITH_PROJECT"]
os.environ["LANGCHAIN_API_KEY"] = os.environ["LANGSMITH_API_KEY"]

In [87]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    base_url="https://router.huggingface.co/v1",
    model="stepfun-ai/Step-3.5-Flash:featherless-ai",
    temperature=0.0
)

In [88]:
llm.invoke("Test")

AIMessage(content="Hello! I'm here and ready to help. 😊  \nWhat would you like to test or discuss today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 14, 'total_tokens': 159, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'stepfun-ai/Step-3.5-Flash', 'system_fingerprint': '', 'id': '63fd3380-5b35-4799-8962-dbbf56e6a398', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--019cbe79-60b7-7e33-ad8b-6e9f10f367e2-0', usage_metadata={'input_tokens': 14, 'output_tokens': 145, 'total_tokens': 159, 'input_token_details': {}, 'output_token_details': {}})

Add the tools,

In [66]:
@tool
def add(a: int, b: int) -> int:
    """
    Add a and b.
    
    Args:
        a (int): first integer to be added
        b (int): second integer to be added

    Return:
        int: sum of a and b
    """
    return a + b

@tool
def subtract(a: int, b:int) -> int:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: int, b:int) -> int:
    """Multiply a and b."""
    return a * b

In [67]:
tool_map = {
    "add": add, 
    "subtract": subtract,
    "multiply": multiply
}

input_ = {
    "a": 1,
    "b": 2
}

tool_map["add"].invoke(input_)

3

In [68]:
tools = [add, subtract, multiply]

llm_with_tools = llm.bind_tools(tools)

Craft the user query,

In [69]:
query = "What is 3 + 2?"
chat_history = [HumanMessage(content=query)]

Invoke the model,

In [70]:
response_1 = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

print(type(response_1))
#print(response_1)

<class 'langchain_core.messages.ai.AIMessage'>


Parse tool calls,

In [71]:
tool_calls_1 = response_1.tool_calls

tool_1_name = tool_calls_1[0]["name"]
tool_1_args = tool_calls_1[0]["args"]
tool_call_1_id = tool_calls_1[0]["id"]

print(f'tool name:\n{tool_1_name}')
print(f'tool args:\n{tool_1_args}')
print(f'tool call ID:\n{tool_call_1_id}')

tool name:
add
tool args:
{'a': 3, 'b': 2}
tool call ID:
call_0


Generate the Tool message,

In [72]:
tool_response = tool_map[tool_1_name].invoke(tool_1_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_1_id)

print(tool_message)

content='5' tool_call_id='call_0'


In [73]:
chat_history.append(tool_message)

Generate the final answer,

In [74]:
answer = llm_with_tools.invoke(chat_history)
print(type(answer))
print(answer.content)

<class 'langchain_core.messages.ai.AIMessage'>
5


In [75]:
for item in chat_history:
    print("-" * 15)
    print(type(item))
    print(item)

---------------
<class 'langchain_core.messages.human.HumanMessage'>
content='What is 3 + 2?' additional_kwargs={} response_metadata={}
---------------
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'tool_calls': [{'id': 'call_0', 'function': {'arguments': '{"a":3, "b":2}', 'name': 'add'}, 'type': 'function', 'index': 0}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 416, 'total_tokens': 498, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'stepfun-ai/Step-3.5-Flash', 'system_fingerprint': '', 'id': 'a285b741-4475-4fb4-bda9-2505f2f00d85', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None} id='run--019cbe58-5dd1-7812-a6bb-dea39781e870-0' tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 2}, 'id': 'call_0', 'type': 'tool_call'}] usage_metadata={'input_tokens': 416, 'output_tokens': 82, 'total_tokens': 498, 'input_token_details': {}, 'output_token_details': {

In [76]:
answer

AIMessage(content='5', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 464, 'total_tokens': 519, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'stepfun-ai/Step-3.5-Flash', 'system_fingerprint': '', 'id': '9fd9edc8-05ac-4220-8d96-9806e332ef20', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--019cbe58-6629-7c41-ab97-5337ddf12ebb-0', usage_metadata={'input_tokens': 464, 'output_tokens': 55, 'total_tokens': 519, 'input_token_details': {}, 'output_token_details': {}})

##### Building the Agent

In [ ]:
class ToolCallingAgent:
    def __init__(self, llm):
        self.llm_with_tools = llm.bind_tools(tools)
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        # Step 1: Initial user message
        chat_history = [HumanMessage(content=query)]

        # Step 2: LLM chooses tool
        response = self.llm_with_tools.invoke(chat_history)
        if not response.tool_calls:
            return response.content # Direct response, no tool needed
        # Step 3: Handle first tool call
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # Step 4: Call tool manually
        tool_result = self.tool_map[tool_name].invoke(tool_args)

        # Step 5: Send result back to LLM
        tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
        chat_history.extend([response, tool_message])

        # Step 6: Final LLM result
        final_response = self.llm_with_tools.invoke(chat_history)
        return final_response.content

In [90]:
my_agent = ToolCallingAgent(llm)

print(my_agent.run("one plus 2"))
print(my_agent.run("one - 2"))
print(my_agent.run("three times two"))

The result is 3.
The result of 1 - 2 is -1.
The result of three times two is **6**.


### 📝 Exercise 1: Create a new tool

Use the example tool format provided in the notebook to create a new tool named `calculate_tip` that takes a `total_bill and tip_percent`, and returns the tip amount. </br>
Define and invoke the tool with sample inputs like `total_bill=120`, `tip_percent=15`. </br>
Create a `tool_map` with the `calculate_tip` tool.


In [96]:
# TODO: Exercise 1
@tool
def calculate_tip(total_bill: float, tip_percent: float) -> float:
    """
    Calculate the tip amount for bill.
    
    Args:
        total_bill (float): Total bill amount
        tip_percent (float): Tip percentage in %

    Return:
        float: Tip amount
    """
    return total_bill * (tip_percent/100)

tool_map = {
    "add": add, 
    "subtract": subtract,
    "multiply": multiply,
    "calculate_tip": calculate_tip
}

tool_map["calculate_tip"].invoke(dict(total_bill=120, tip_percent=15))

18.0

### 📝 Exercise 2: Tool calling with an LLM

Simulate a user query like "How much should I tip on $60 at 20%?". </br>
Bind the tool to the predefined `llm` and prompt the LLM with the query above. Then parse the LLM response for the tool calling details and invoke the tool accordingly. Finally, take the entire chat history and prompt the LLM for a final output.


In [100]:
# TODO: Exercise 2
query = "How much should I tip on $60 at 20%?"
chat_history = [HumanMessage(content=query)]

llm = ChatOpenAI(
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    base_url="https://router.huggingface.co/v1",
    model="stepfun-ai/Step-3.5-Flash:featherless-ai",
    temperature=0.0
)

llm_with_tools = llm.bind_tools(list(tool_map.values()))

response_1 = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

tool_calls_1 = response_1.tool_calls
tool_1_args = tool_calls_1[0]["args"]
tool_call_1_id = tool_calls_1[0]["id"]

tool_response = tool_map["calculate_tip"].invoke(tool_1_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_1_id)
chat_history.append(tool_message)

answer = llm_with_tools.invoke(chat_history)
print(type(answer))
print(answer.content)

<class 'langchain_core.messages.ai.AIMessage'>
You should tip $12.00 on a $60 bill at 20%.


### 📝 Exercise 3: Create a tip calculating agent

Create an agent to automate the entire process you previously completed.


In [103]:
# TODO: Exercise 3
query = "How much should I tip on $60 at 20%?"

class ToolCallingAgent:
    def __init__(self, llm):
        self.llm_with_tools = llm.bind_tools(list(tool_map.values()))
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        # Step 1: Initial user message
        chat_history = [HumanMessage(content=query)]

        # Step 2: LLM chooses tool
        response = self.llm_with_tools.invoke(chat_history)
        if not response.tool_calls:
            return response.content # Direct response, no tool needed
        # Step 3: Handle first tool call
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # Step 4: Call tool manually
        tool_result = self.tool_map[tool_name].invoke(tool_args)

        # Step 5: Send result back to LLM
        tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
        chat_history.extend([response, tool_message])

        # Step 6: Final LLM result
        final_response = self.llm_with_tools.invoke(chat_history)
        return final_response.content

my_agent = ToolCallingAgent(llm)
my_agent.run(query)

'You should tip $12.00 on a $60 bill at 20%.'